# GR-Conditioned Output-Transformer LSTM — raw input + diffssl `tvcond` (Diff-SSL)

**Google Colab**: Runtime → **GPU**. Open via *File → Open notebook → GitHub*
(`5aola/Virtual-Analogue-Compressor-Modelling`); cell 1 clones the repo for the
`06_output` modules and mounts Drive for the dataset. **Push local changes before running.**

## Idea — the diffssl SOTA architecture, conditioned on the *ground-truth* GR curve

This is the `02b_sota_training/train_lstm_diffssl_tvc.ipynb` recipe with **one change**:
the time-varying conditioning is generated from the **exported GR curve** instead of
being learned from `|x|` + static knobs.

```
dry (raw) ───────────────────────────────►┐
                                            ├─► [GR-conditioned LSTM] ─► wet
exported GR curve (dB) ─► tvcond generator ─┘   (concat to LSTM input)
```

- **Input** (`INPUT_SIGNAL="raw"`): the raw dry signal — the model learns the gain
  itself (no amplitude matching). Set `"amp"` to fall back to the grey-box
  amplitude-matched input (`dry · 10**(gr_db/20)`).
- **Conditioning** (`use_gr_cond`): the exported `gr_db` curve drives `GRTVCond`
  (block-rate LSTM → upsample → concat to the LSTM input) — the diffssl `tvcond`
  mechanism, GR-driven instead of `|x|`-driven. Three LSTM states are carried across
  TBPTT chunks (the two main LSTMs + the conditioning generator).
- **Output** (`OUTPUT_MODE="direct"`): predict the wet directly, matching `02b`.
  `"residual_gain"` (out = dry·g, g≈1 at init) is a physically-grounded alternative
  if direct training is unstable.

This isolates the question the `02b` run raised: does privileged GR conditioning
**beat** the SOTA's self-learned envelope, or merely **match** it?

## Identical to 02b / the other 06_output notebooks
- **Dataset / split**: Diff-SSL-G-Comp, 10 settings × 10 songs, `splits.py` seed 42
  (val = 1 song × all settings, test = held-out songs × lowest-threshold).
- **Loss / metrics**: `0.5·L1 + 0.5·MR-STFT` (+opt. ESR); ESR/RMSE/MAE/MSE.
- **Training**: fixed 100-epoch budget, no early stopping, cosine LR.


In [ ]:
# -- 0. Dependencies ---------------------------------------------------
# This variant uses nablafx (TVFiLMMod). Pin numpy first so lightning/nablafx
# installs can't downgrade Colab's numpy 2.x and break torch. Install
# lightning/nablafx --no-deps so they can't clobber Colab's CUDA torch.
# `rational` / `frechet_audio_distance` are nablafx import-chain deps we never
# use here; stub both so `from nablafx...` doesn't drag in broken wheels.
!pip install -q "numpy>=2.0,<2.6"
!pip install -q torchmetrics soundfile auraloss einops lightning-utilities packaging
!pip install -q --no-deps lightning nablafx

import sys, types

rational = types.ModuleType("rational")
rational.torch = types.ModuleType("rational.torch")
rational.torch.Rational = type("Rational", (), {})
sys.modules["rational"], sys.modules["rational.torch"] = rational, rational.torch

fad = types.ModuleType("frechet_audio_distance")
fad.FrechetAudioDistance = type("FrechetAudioDistance", (), {})
sys.modules["frechet_audio_distance"] = fad

import numpy as np, torch
assert np.__version__.startswith("2."), f"numpy {np.__version__} - restart runtime, re-run cell 0"
print(f"numpy {np.__version__}, torch {torch.__version__}")


In [ ]:
# -- 1. Mount Drive (dataset) + clone repo from GitHub (code) ---------
# The repo is NOT synced to Drive (only data/ is). Code comes from GitHub -
# push local changes before (re)running this cell; re-running pulls updates.

import os
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

DRIVE_DATA_ROOT = "/content/drive/Othercomputers/MacBook Air/data/Diff-SSL-G-Comp"
REPO_URL = "https://github.com/5aola/Virtual-Analogue-Compressor-Modelling.git"
REPO_ROOT = "/content/Virtual-Analogue-Compressor-Modelling"

if os.path.isdir(REPO_ROOT):
    !git -C "{REPO_ROOT}" pull --ff-only
else:
    !git clone --depth 1 "{REPO_URL}" "{REPO_ROOT}"

DATA_ROOT = DRIVE_DATA_ROOT

# Module directory for this notebook.
# NOTE: Colab clones from GitHub, so any *uncommitted* local folders won't exist there.
# The implementation for this notebook lives in `06_output/`.
CANDIDATE_DIRS = [
    os.path.join(REPO_ROOT, "06_output"),
    os.path.join(REPO_ROOT, "06_conditioning"),  # legacy / local-only fallback
]
COND_DIR = next((d for d in CANDIDATE_DIRS if os.path.isfile(os.path.join(d, "dataset.py"))), None)
assert COND_DIR is not None, (
    f"Could not find dataset.py in any of: {CANDIDATE_DIRS}. "
    f"Did the clone succeed? REPO_ROOT={REPO_ROOT}"
)

OUTPUT_DIR = os.path.join(os.path.dirname(DATA_ROOT), "output_transformer_tfilm_runs")

assert os.path.isdir(os.path.join(DATA_ROOT, "gr_curves")), f"Bad DATA_ROOT: {DATA_ROOT}"
assert os.path.isdir(os.path.join(DATA_ROOT, "processed_ground_truth")), "Missing wet audio dir"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# repo root (for `src`) + module dir (for dataset/model/system/splits)
for p in (REPO_ROOT, COND_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"REPO_ROOT  : {REPO_ROOT}")
print(f"COND_DIR   : {COND_DIR}")
print(f"DATA_ROOT  : {DATA_ROOT}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")


In [ ]:
# -- 2. Cache dataset to Colab local SSD ------------------------------
# The output transformer needs dry + GR curves (to build the matched input)
# AND the wet audio (the target). Mirror 05's cache and add the wet WAVs.

import shutil
from dataset import discover_output_transformer_pairs

LOCAL_DATA_ROOT = "/content/Diff-SSL-G-Comp"

pairs = discover_output_transformer_pairs(DATA_ROOT)
settings = sorted({p["setting"] for p in pairs})
songs = sorted({p["song"] for p in pairs})
print(f"Caching {len(songs)} songs x {len(settings)} settings ({len(pairs)} pairs) -> {LOCAL_DATA_ROOT}")

def _mirror(src: Path, dst: Path):
    src, dst = Path(src), Path(dst)
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)

# dry WAVs (one per song, shared across settings)
for song in songs:
    fn = f"{song}_UnmasteredWAV.wav"
    _mirror(Path(DATA_ROOT) / "processed_normalized" / fn,
            Path(LOCAL_DATA_ROOT) / "processed_normalized" / fn)

# GR curves (.pt) + wet WAVs (-exported.wav), per (song, setting) pair
for p in pairs:
    _mirror(p["gr"], Path(LOCAL_DATA_ROOT) / "gr_curves" / p["setting"] / Path(p["gr"]).name)
    _mirror(p["wet"], Path(LOCAL_DATA_ROOT) / "processed_ground_truth" / p["setting"] / Path(p["wet"]).name)

DATA_ROOT = LOCAL_DATA_ROOT
print(f"Using local cache: {DATA_ROOT}")


In [ ]:
# -- 3. Imports & hyper-parameters ------------------------------------

import json
from datetime import datetime

import torch
import lightning as pl
from lightning.pytorch.callbacks import (
    LearningRateMonitor, ModelCheckpoint, TQDMProgressBar,
)
from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger

from dataset import SAMPLE_RATE, SEGMENT_LEN, WINDOW, discover_output_transformer_pairs
from dataset_tfilm import OutputTransformerTFiLMDataModule
from model_tfilm import OutputTransformerTFiLMLSTM
from system_tfilm import OutputTransformerTFiLMSystem
from splits import build_split_manifest
from amplitude_match import GR_DB_MIN, GR_DB_MAX

print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "WARNING: CPU runtime")

# -- split (identical to 05 / 06) --
SPLIT_SEED   = 42
N_VAL_SONGS  = 1
N_TEST_SONGS = 2

# -- training (aligned to diffssl LSTM32TVC baseline: fixed-epoch budget, no
#    early stopping. Only deviation per request: keep cosine LR instead of
#    diffssl's ReduceLROnPlateau.) --
LR             = 1e-3
MAX_EPOCHS     = 100       # fixed budget (matches diffssl) == CosineAnnealingLR T_max
SCHEDULER      = "cosine"  # kept per request (diffssl uses ReduceLROnPlateau)
ETA_MIN        = 1e-6
WARMUP_SAMPLES = 0         # drop N samples on each track's fresh-state chunk

# -- input / output mode --
# INPUT_SIGNAL selects the model's WAVEFORM input:
#   "raw" : raw dry signal -> the model learns the gain itself from the GR
#           conditioning (matches the 02b diffssl SOTA input).
#   "amp" : amplitude-matched dry (grey-box front-end, dry * 10**(gr_db/20)).
INPUT_SIGNAL = "raw"

# -- model (GR-conditioned sample-rate windowed LSTM) --
ENCODER_CHANNELS = 8
HIDDEN_SIZE      = 16
MID_CHANNELS     = 8
NUM_LSTM_LAYERS  = 2
# "direct" predicts the wet from scratch (matches 02b "Direct Output"); use
# "residual_gain" (out = input * g, g~1 at init) for a physically-grounded,
# faster-converging alternative. "residual_add" only makes sense with "amp".
OUTPUT_MODE      = "direct"
OUT_ACTIVATION   = "none"           # "tanh" to bound the output

# -- GR conditioning (diffssl tvcond mechanism, GR-driven) --
# Same mechanism as diffssl LSTM32TVC: a block-rate LSTM emits a cond_dim-wide
# time-varying sequence that is upsampled and CONCATENATED to the LSTM input.
# Difference vs diffssl: the generator reads the exported GR curve (which IS the
# envelope diffssl had to learn from |x|+knobs) instead of |x|.
USE_GR_COND     = True   # tvcond conditioning on/off (off = plain raw-in LSTM)
COND_DIM        = 16     # width of the generated conditioning sequence (diffssl: 16)
COND_BLOCK_SIZE = 128    # samples/block (diffssl: 128 -> 128/44100 ~= 2.9 ms)
COND_NUM_LAYERS = 1      # block-rate conditioning LSTM depth (diffssl: 1)

# -- loss (SOTA waveform recipe) --
L1_WEIGHT     = 0.5
MRSTFT_WEIGHT = 0.5
ESR_WEIGHT    = 0.0   # set > 0 to add the Optical-DRC / comparative-study ESR term

RUN_TAG    = f"lstm_ot_{INPUT_SIGNAL}in_gr_tvcond"
RESUME_RUN = None


In [ ]:
# -- 4. Preview split (must match 05) ---------------------------------

preview = build_split_manifest(
    discover_output_transformer_pairs(DATA_ROOT),
    seed=SPLIT_SEED, n_val_songs=N_VAL_SONGS, n_test_songs=N_TEST_SONGS,
)
print(f"Settings ({len(preview.all_settings)}): {preview.all_settings}")
print(f"Test settings (lowest T): {preview.test_settings}")
print(f"Train songs: {preview.train_songs}")
print(f"Val songs  : {preview.val_songs}")
print(f"Test songs : {preview.test_songs}")
print(f"Pairs - train={len(preview.train_pair_keys)} "
      f"val={len(preview.val_pair_keys)} test={len(preview.test_pair_keys)}")


In [ ]:
# -- 5. Model size ----------------------------------------------------

model = OutputTransformerTFiLMLSTM(
    window=WINDOW, encoder_channels=ENCODER_CHANNELS, hidden_size=HIDDEN_SIZE,
    mid_channels=MID_CHANNELS, num_lstm_layers=NUM_LSTM_LAYERS,
    output_mode=OUTPUT_MODE, out_activation=OUT_ACTIVATION,
    use_gr_cond=USE_GR_COND, cond_dim=COND_DIM,
    cond_block_size=COND_BLOCK_SIZE, cond_num_layers=COND_NUM_LAYERS,
)
n_params = sum(p.numel() for p in model.parameters())
print(f"OutputTransformerTFiLMLSTM: {n_params:,} params  "
      f"(use_gr_cond={USE_GR_COND}, cond_dim={COND_DIM}, output_mode={OUTPUT_MODE})")
for name, mod in model.named_children():
    print(f"  {name:14s} {sum(p.numel() for p in mod.parameters()):,}")
print(f"\nWindow {WINDOW} samples | segment {SEGMENT_LEN} ({SEGMENT_LEN/SAMPLE_RATE:.2f}s) | "
      f"{SAMPLE_RATE} Hz | cond block {COND_BLOCK_SIZE} "
      f"({COND_BLOCK_SIZE/SAMPLE_RATE*1e3:.1f} ms)")


In [ ]:
# -- 6. Train ---------------------------------------------------------

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

assert DATA_ROOT.startswith("/content/"), "Run the cache cell first (cell 2)."

if RESUME_RUN:
    RUN_NAME = RESUME_RUN
    RUN_DIR = os.path.join(OUTPUT_DIR, RUN_NAME)
    _resume_ckpt = os.path.join(RUN_DIR, "checkpoints", "last.ckpt")
    print(f"RESUMING: {RUN_NAME}")
else:
    RUN_NAME = f"ot_lstm_{datetime.now():%Y%m%d_%H%M%S}_{RUN_TAG}"
    RUN_DIR = os.path.join(OUTPUT_DIR, RUN_NAME)
    _resume_ckpt = None
    print(f"NEW run: {RUN_NAME}")

os.makedirs(RUN_DIR, exist_ok=True)
split_path = os.path.join(RUN_DIR, "split_manifest.json")

dm = OutputTransformerTFiLMDataModule(
    data_root=DATA_ROOT, segment_len=SEGMENT_LEN, window=WINDOW,
    sample_rate=SAMPLE_RATE, split_seed=SPLIT_SEED,
    n_val_songs=N_VAL_SONGS, n_test_songs=N_TEST_SONGS,
    split_manifest_path=split_path,
    input_signal=INPUT_SIGNAL,
)
dm.setup()
print(f"Train/val/test streams: {dm.train_dataset.B} / {dm.val_dataset.B} / {dm.test_dataset.B}")
print(f"Steps/epoch (train): {len(dm.train_dataset)}")

with open(os.path.join(RUN_DIR, "hparams.json"), "w") as f:
    json.dump({
        "approach": f"gr_output_transformer_{INPUT_SIGNAL}_input_gr_conditioned",
        "model_type": "sample_rate_windowed_lstm",
        "source_model": "Optical-DRC create_model_LSTM + PReLU front-end; diffssl tvcond conditioning",
        "dataset": "Diff-SSL-G-Comp", "settings": "all 10 (pooled, no static-knob conditioning)",
        "input": ("raw dry (GR used only as conditioning)" if INPUT_SIGNAL == "raw"
                  else "amplitude_matched = dry * 10**(clamp(gr_db, %g, %g)/20)" % (GR_DB_MIN, GR_DB_MAX)),
        "input_signal": INPUT_SIGNAL,
        "conditioning": "exported GR curve via diffssl tvcond (block-rate LSTM -> concat to LSTM input)",
        "target": "wet audio (direct)", "sample_rate": SAMPLE_RATE,
        "window": WINDOW, "segment_len": SEGMENT_LEN,
        "split_seed": SPLIT_SEED, "train_songs": dm.split.train_songs,
        "val_songs": dm.split.val_songs, "test_songs": dm.split.test_songs,
        "test_settings": dm.split.test_settings,
        "model": {"encoder_channels": ENCODER_CHANNELS, "hidden_size": HIDDEN_SIZE,
                   "mid_channels": MID_CHANNELS, "num_lstm_layers": NUM_LSTM_LAYERS,
                   "output_mode": OUTPUT_MODE, "out_activation": OUT_ACTIVATION,
                   "use_gr_cond": USE_GR_COND, "cond_dim": COND_DIM,
                   "cond_block_size": COND_BLOCK_SIZE, "cond_num_layers": COND_NUM_LAYERS,
                   "num_params": n_params},
        "loss": {"l1": L1_WEIGHT, "mrstft": MRSTFT_WEIGHT, "esr": ESR_WEIGHT,
                  "ref": "nablafx-diffssl 0.5*L1+0.5*MR-STFT (+opt ESR)"},
        "metrics": ["esr", "rmse", "mae", "mse"],
        "lr": LR, "max_epochs": MAX_EPOCHS, "scheduler": SCHEDULER,
        "eta_min": ETA_MIN, "warmup_samples": WARMUP_SAMPLES,
    }, f, indent=2)

system = OutputTransformerTFiLMSystem(
    model=model, lr=LR, l1_weight=L1_WEIGHT, mrstft_weight=MRSTFT_WEIGHT,
    esr_weight=ESR_WEIGHT, warmup_samples=WARMUP_SAMPLES, scheduler=SCHEDULER,
    max_epochs=MAX_EPOCHS, eta_min=ETA_MIN,
)

ckpt_dir = os.path.join(RUN_DIR, "checkpoints")
callbacks = [
    ModelCheckpoint(dirpath=ckpt_dir, monitor="loss/val", mode="min", save_top_k=3,
                    save_last=True, filename="best-{epoch:03d}-{step}",
                    auto_insert_metric_name=False),
    LearningRateMonitor(logging_interval="epoch"),
    TQDMProgressBar(refresh_rate=10),
]
loggers = [
    TensorBoardLogger(save_dir=RUN_DIR, name="tb", version=""),
    CSVLogger(save_dir=RUN_DIR, name="csv", version=""),
]

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS, accelerator="gpu", devices=1,
    callbacks=callbacks, logger=loggers,
    log_every_n_steps=10,
)
trainer.fit(system, dm, ckpt_path=_resume_ckpt)
print(f"Best val loss: {callbacks[0].best_model_score:.6f}")
print(f"Best ckpt    : {callbacks[0].best_model_path}")


In [ ]:
# -- 7. Test (held-out songs x lowest-threshold settings) -------------

trainer.test(system, datamodule=dm, ckpt_path=callbacks[0].best_model_path)


In [ ]:
# -- 8. Plot: model input vs prediction vs target --------------------
# Streams the val set in order so the LSTM state settles, then plots a
# mid-track chunk per stream. With INPUT_SIGNAL="raw" the grey trace is the
# raw dry input; with "amp" it is the amplitude-matched grey-box baseline.

import matplotlib.pyplot as plt
import numpy as np
from system import esr_metric, _detach_state

best = torch.load(callbacks[0].best_model_path, map_location="cuda", weights_only=False)
system.load_state_dict(best["state_dict"])
system.eval().cuda()
print(f"Loaded best checkpoint: {callbacks[0].best_model_path}")

val_steps = list(dm.val_dataloader())
pick = len(val_steps) // 2

state = None
with torch.no_grad():
    for s, (inp, gr, wet, mask, reset) in enumerate(val_steps):
        if bool(reset):
            state = None
        pred, state = system.model(inp.cuda(), gr.cuda(), state, return_state=True)
        state = _detach_state(state)
        if s == pick:
            amp = inp[:, :, WINDOW - 1:].cpu().numpy()   # model input (raw or amp)
            pred_np = pred.cpu().numpy()
            wet_np = wet.numpy()
            rows = torch.nonzero(mask).squeeze(1).tolist()
            break

in_label = "raw dry (model input)" if INPUT_SIGNAL == "raw" else "Amplitude-matched (baseline in)"
n_plots = min(4, len(rows))
fig, axes = plt.subplots(n_plots, 1, figsize=(14, 3 * n_plots), sharex=True, squeeze=False)
t = np.arange(wet_np.shape[-1]) / SAMPLE_RATE
for ax, r in zip(axes[:, 0], rows[:n_plots]):
    ax.plot(t, amp[r, 0], label=in_label, alpha=0.4, lw=0.5, color="gray")
    ax.plot(t, wet_np[r, 0], label="Target (wet)", alpha=0.8, lw=0.5)
    ax.plot(t, pred_np[r, 0], label="Predicted", alpha=0.8, lw=0.5)
    base_mae = float(np.mean(np.abs(amp[r, 0] - wet_np[r, 0])))
    pred_mae = float(np.mean(np.abs(pred_np[r, 0] - wet_np[r, 0])))
    pv = torch.from_numpy(pred_np[r]); tv = torch.from_numpy(wet_np[r])
    c = dm.val_dataset.cache[r]
    ax.set_title(f"{c['song']} / {c['setting']} (chunk {pick}) - "
                 f"MAE: input {base_mae:.4f} -> pred {pred_mae:.4f} | ESR {float(esr_metric(tv, pv)):.4f}")
    ax.set_ylabel("amp"); ax.legend(loc="lower right", fontsize=8); ax.set_ylim(-1.05, 1.05)
axes[-1, 0].set_xlabel("Time (s)")
fig.suptitle(f"GR-conditioned output-transformer LSTM ({INPUT_SIGNAL} in) - best val loss {callbacks[0].best_model_score:.6f}", y=1.005)
fig.tight_layout()
plot_path = os.path.join(RUN_DIR, "eval_output_comparison.png")
fig.savefig(plot_path, dpi=150, bbox_inches="tight")
print(f"Saved plot -> {plot_path}")
plt.show()


In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{RUN_DIR}/tb"
